In [0]:
from pyspark.sql.functions import (
    col, lower, trim, regexp_replace, when,
    hour, month, dayofweek
)

bronze_table = "workspace.default.bronze_accidents_hpms"
silver_table = "workspace.default.silver_accidents_hpms"

df = spark.table(bronze_table)
print(f"Loaded {df.count()} rows from Bronze")

In [0]:
# Extract time features from Start_Time
df = df.withColumn("hour", hour(col("Start_Time"))) \
       .withColumn("month", month(col("Start_Time"))) \
       .withColumn("day_of_week", dayofweek(col("Start_Time")))

print("Time features extracted")

In [0]:
critical_cols = [
    "Severity", "Start_Time", "Start_Lat", "Start_Lng",
    "Temperature_F", "Visibility_mi", "SPEED_LIMIT", "AADT"
]
df = df.dropna(subset=critical_cols)

In [0]:
# Filter top 25 weather conditions
top_25 = [r[0] for r in df.groupBy("Weather_Condition").count()
          .orderBy(col("count").desc()).limit(25).collect()]
df = df.filter(col("Weather_Condition").isin(top_25))

# Standardize weather condition labels
df = df.withColumn("Weather_Condition", lower(trim(
    regexp_replace(col("Weather_Condition"), r'[/\\-]', ' '))))
df = df.withColumn("Weather_Condition",
    regexp_replace(col("Weather_Condition"), r'\s+', ' '))

df = df.withColumn("Weather_Condition",
    when(col("Weather_Condition").rlike('light rain with thunder|thunder|t storm'), 'Rainy_Thunder')
    .when(col("Weather_Condition").rlike('light rain|light drizzle|rain'), 'Rain_Drizzle')
    .when(col("Weather_Condition").rlike('light snow|snow'), 'Snow')
    .when(col("Weather_Condition").rlike('fair|clear'), 'Clear_Fair')
    .when(col("Weather_Condition").rlike('mostly cloudy windy|cloudy windy'), 'Cloudy_Windy')
    .when(col("Weather_Condition").rlike('mostly cloudy|cloudy|partly cloudy|overcast|scattered clouds'), 'Cloudy')
    .otherwise(col("Weather_Condition"))
)
print("Weather conditions standardized")

In [0]:
# Remove outliers using IQR for weather + HPMS columns
def get_iqr_bounds(df, column, lower_multiplier=1.5, upper_multiplier=1.5):
    quantiles = df.select(col(column).cast("float").alias("temp_col")) \
                  .approxQuantile("temp_col", [0.25, 0.75], 0.05)
    iqr = quantiles[1] - quantiles[0]
    return quantiles[0] - (lower_multiplier * iqr), quantiles[1] + (upper_multiplier * iqr)

wind_lower, wind_upper = get_iqr_bounds(df, "Wind_Speed_mph")
temp_lower, temp_upper = get_iqr_bounds(df, "Temperature_F",
                                         lower_multiplier=3.0, upper_multiplier=1.5)
aadt_lower, aadt_upper = get_iqr_bounds(df, "AADT")
speed_lower, speed_upper = get_iqr_bounds(df, "SPEED_LIMIT")

df = df.filter(
    (col("Wind_Speed_mph") >= wind_lower) & (col("Wind_Speed_mph") <= wind_upper) &
    (col("Temperature_F") >= temp_lower) & (col("Temperature_F") <= temp_upper) &
    (col("AADT") >= aadt_lower) & (col("AADT") <= aadt_upper) &
    (col("SPEED_LIMIT") >= speed_lower) & (col("SPEED_LIMIT") <= speed_upper)
)
print(f"After outlier removal: {df.count()} rows")

In [0]:
# Save as Silver Delta Table
df.write.format("delta").mode("overwrite").saveAsTable(silver_table)
print(f"Silver layer done! Rows: {df.count()}")
df.show(5)

In [0]:
# Verify
spark.sql("SELECT * FROM workspace.default.silver_accidents_hpms LIMIT 5").show()